In [4]:
# Check variables

print("X_train exists:", "X_train" in globals())
print("y_train exists:", "y_train" in globals())
print("targets exists:", "targets" in globals())

if "X_train" in globals():
    print("X_train shape:", X_train.shape)

if "y_train" in globals():
    print("y_train shape:", y_train.shape)

if "targets" in globals():
    print("targets:", targets)

X_train exists: True
y_train exists: False
targets exists: True
X_train shape: (12389, 22)
targets: ['Temp', 'RH', 'Rain', 'PRESS', 'SUNSHINE']


In [12]:
# =====================================================
# GRU BASELINE MODEL
# =====================================================

from tensorflow.keras.layers import Input, GRU, Dense, Embedding, GlobalMaxPooling1D, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# =====================================================
# BUILD GRU MODEL
# =====================================================

num_features = X_train.shape[2]
if len(y_train.shape) == 1:
    y_train = y_train.reshape(-1, 1)
    y_val = y_val.reshape(-1, 1)
    y_test = y_test.reshape(-1, 1)

num_targets = y_train.shape[1]
num_stations = df["Station_enc"].nunique()

seq_input = Input(shape=(SEQ_LEN, num_features), name="sequence_input")

x = GRU(
    units=64,
    return_sequences=False,
    name="gru_layer"
)(seq_input)

station_input = Input(shape=(1,), name="station_input")

s = Embedding(
    input_dim=num_stations,
    output_dim=4,
    name="station_embedding"
)(station_input)

s = GlobalMaxPooling1D()(s)

combined = Concatenate()([x, s])

z = Dense(64, activation="relu")(combined)
z = Dropout(0.2)(z)

output = Dense(num_targets, activation="linear")(z)

gru_model = Model(
    inputs=[seq_input, station_input],
    outputs=output
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

gru_model.summary()


# =====================================================
# TRAIN GRU MODEL
# =====================================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history_gru = gru_model.fit(
    [X_train, S_train],
    y_train,
    validation_data=([X_val, S_val], y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# =====================================================
# PREDICTION
# =====================================================

y_pred_scaled = gru_model.predict([X_test, S_test])

y_pred_gru = Y_scaler.inverse_transform(y_pred_scaled)
y_true_gru = Y_scaler.inverse_transform(y_test)


# If you used log-rain transformation
rain_idx = targets.index("Rain")

y_pred_gru[:, rain_idx] = np.expm1(y_pred_gru[:, rain_idx])
y_true_gru[:, rain_idx] = np.expm1(y_true_gru[:, rain_idx])


# =====================================================
# EVALUATION
# =====================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

gru_results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(y_true_gru[:, i], y_pred_gru[:, i])
    mse = mean_squared_error(y_true_gru[:, i], y_pred_gru[:, i])
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_gru[:, i], y_pred_gru[:, i])

    gru_results.append({
        "Model": "GRU",
        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

gru_results_df = pd.DataFrame(gru_results)

print(gru_results_df)

gru_results_df.to_excel(
    "GRU_Results.xlsx",
    index=False
)

print("Saved: GRU_Results.xlsx")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequence_input      │ (None, 30, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_embedding   │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_layer (GRU)     │ (None, 64)        │     14,400 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ station_embeddin… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 68)        │          0 │ gru_layer[0][0],  │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,416 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │        325 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,161 (74.85 KB)

 Trainable params: 19,161 (74.85 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0287 - mae: 0.1169 - val_loss: 0.0124 - val_mae: 0.0732
Epoch 2/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0139 - mae: 0.0813 - val_loss: 0.0110 - val_mae: 0.0648
Epoch 3/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0125 - mae: 0.0747 - val_loss: 0.0103 - val_mae: 0.0604
Epoch 4/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0115 - mae: 0.0702 - val_loss: 0.0097 - val_mae: 0.0579
Epoch 5/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0110 - mae: 0.0680 - val_loss: 0.0096 - val_mae: 0.0581
Epoch 6/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0105 - mae: 0.0656 - val_loss: 0.0095 - val_mae: 0.0598
Epoch 7/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0102 - mae: 0.0647 - val_loss: 0.0094 - val_mae: 0.0572
Epoch 8/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0099 - mae: 0.0631 - val_loss: 0.0092 - val_mae: 0.0559
Epoch 9/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/

In [11]:
# =====================================================
# PREPARE MODEL PARAMETERS
# =====================================================

num_features = X_train.shape[2]
num_targets = y_train.shape[1]
num_stations = df["Station_enc"].nunique()

print("Features:", num_features)
print("Targets:", num_targets)
print("Stations:", num_stations)

# =====================================================
# BUILD GRU MODEL
# =====================================================

from tensorflow.keras.layers import (
    Input, GRU, Dense, Embedding,
    GlobalMaxPooling1D, Concatenate, Dropout
)

from tensorflow.keras.models import Model

seq_input = Input(
    shape=(SEQ_LEN, num_features),
    name="sequence_input"
)

x = GRU(
    units=64,
    return_sequences=False
)(seq_input)

station_input = Input(
    shape=(1,),
    name="station_input"
)

s = Embedding(
    input_dim=num_stations,
    output_dim=4
)(station_input)

s = GlobalMaxPooling1D()(s)

combined = Concatenate()([x, s])

z = Dense(64, activation="relu")(combined)
z = Dropout(0.2)(z)

output = Dense(
    num_targets,
    activation="linear"
)(z)

gru_model = Model(
    inputs=[seq_input, station_input],
    outputs=output
)

Features: 9
Targets: 5
Stations: 5


In [4]:
print("df exists:", "df" in globals())
print("train_df exists:", "train_df" in globals())
print("X_train exists:", "X_train" in globals())
print("y_train exists:", "y_train" in globals())
print("S_train exists:", "S_train" in globals())

df exists: False
train_df exists: False
X_train exists: False
y_train exists: False
S_train exists: False


In [5]:
# 1. Load dataset
import pandas as pd
import numpy as np

df = pd.read_excel("final_filled_climate_cleaned.xlsx")

df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)

df = df.dropna(subset=["Date"])
df = df.sort_values(["Station", "Date"]).reset_index(drop=True)

In [6]:
# 2. Time features
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year

In [10]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Station_enc"] = le.fit_transform(df["Station"])

print(df[["Station", "Station_enc"]].drop_duplicates())



df["Station_enc"]

# =====================================================
# 6. DEFINE FEATURES AND TARGETS
# =====================================================

features = [
    "Temp",
    "RH",
    "Wind",
    "Rain",
    "PRESS",
    "SUNSHINE",
    "Month",
    "Week",
    "DayOfWeek"
]

targets = [
    "Temp",
    "RH",
    "Rain",
    "PRESS",
    "SUNSHINE"
]


# =====================================================
# 7. CHRONOLOGICAL SPLIT BEFORE SCALING
# =====================================================

train_df = df[df["Year"] <= 2022].copy()
val_df   = df[df["Year"] == 2023].copy()
test_df  = df[df["Year"] == 2024].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# =====================================================
# 8. SCALE DATA
# =====================================================

X_scaler = MinMaxScaler()
Y_scaler = MinMaxScaler()

X_scaler.fit(train_df[features])
Y_scaler.fit(train_df[targets])

train_X = X_scaler.transform(train_df[features])
val_X   = X_scaler.transform(val_df[features])
test_X  = X_scaler.transform(test_df[features])

train_Y = Y_scaler.transform(train_df[targets])
val_Y   = Y_scaler.transform(val_df[targets])
test_Y  = Y_scaler.transform(test_df[targets])

train_station = train_df["Station_enc"].values
val_station   = val_df["Station_enc"].values
test_station  = test_df["Station_enc"].values


# =====================================================
# 9. CREATE 30-DAY SEQUENCES
# =====================================================

SEQ_LEN = 30

def create_sequences(X, Y, station_ids, dates, seq_len=30):
    X_seq = []
    S_seq = []
    Y_seq = []
    D_seq = []

    for station in np.unique(station_ids):
        idx = np.where(station_ids == station)[0]

        X_station = X[idx]
        Y_station = Y[idx]
        dates_station = dates.iloc[idx].reset_index(drop=True)

        for i in range(len(X_station) - seq_len):
            X_seq.append(X_station[i:i + seq_len])
            S_seq.append(station)
            Y_seq.append(Y_station[i + seq_len])
            D_seq.append(dates_station.iloc[i + seq_len])

    return np.array(X_seq), np.array(S_seq), np.array(Y_seq), np.array(D_seq)


X_train, S_train, y_train, d_train = create_sequences(
    train_X, train_Y, train_station, train_df["Date"], SEQ_LEN
)

X_val, S_val, y_val, d_val = create_sequences(
    val_X, val_Y, val_station, val_df["Date"], SEQ_LEN
)

X_test, S_test, y_test, d_test = create_sequences(
    test_X, test_Y, test_station, test_df["Date"], SEQ_LEN
)

print("X_train:", X_train.shape)
print("S_train:", S_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("X_test:", X_test.shape)



        Station  Station_enc
0      HEB00008            0
3279   JEN00001            1
6586   JER00005            2
9971   NAB00003            3
13248  RAM00004            4
Train: (12539, 13)
Validation: (1814, 13)
Test: (1818, 13)
X_train: (12389, 30, 9)
S_train: (12389,)
y_train: (12389, 5)
X_val: (1664, 30, 9)
X_test: (1668, 30, 9)


In [14]:
# =====================================================
# CONVLSTM BASELINE MODEL
# =====================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Input, ConvLSTM2D, Flatten, Dense,
    Embedding, GlobalMaxPooling1D,
    Concatenate, Dropout
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =====================================================
# 1. RESHAPE DATA FOR CONVLSTM
# =====================================================

# Original:
# X_train shape = (samples, 30, 9)

X_train_conv = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1,
    X_train.shape[2],
    1
)

X_val_conv = X_val.reshape(
    X_val.shape[0],
    X_val.shape[1],
    1,
    X_val.shape[2],
    1
)

X_test_conv = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1,
    X_test.shape[2],
    1
)

print("X_train_conv:", X_train_conv.shape)
print("X_val_conv:", X_val_conv.shape)
print("X_test_conv:", X_test_conv.shape)


# =====================================================
# 2. MODEL PARAMETERS
# =====================================================

SEQ_LEN = X_train.shape[1]
num_features = X_train.shape[2]
num_targets = y_train.shape[1]
num_stations = df["Station_enc"].nunique()


# =====================================================
# 3. BUILD CONVLSTM MODEL
# =====================================================

seq_input = Input(
    shape=(SEQ_LEN, 1, num_features, 1),
    name="convlstm_input"
)

x = ConvLSTM2D(
    filters=64,
    kernel_size=(1, 3),
    activation="relu",
    padding="same",
    return_sequences=False
)(seq_input)

x = Flatten()(x)

station_input = Input(
    shape=(1,),
    name="station_input"
)

s = Embedding(
    input_dim=num_stations,
    output_dim=4,
    name="station_embedding"
)(station_input)

s = GlobalMaxPooling1D()(s)

combined = Concatenate()([x, s])

z = Dense(64, activation="relu")(combined)
z = Dropout(0.2)(z)

output = Dense(
    num_targets,
    activation="linear"
)(z)

convlstm_model = Model(
    inputs=[seq_input, station_input],
    outputs=output
)

convlstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

convlstm_model.summary()


# =====================================================
# 4. TRAIN CONVLSTM
# =====================================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history_convlstm = convlstm_model.fit(
    [X_train_conv, S_train],
    y_train,
    validation_data=([X_val_conv, S_val], y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# =====================================================
# 5. PREDICTION
# =====================================================

y_pred_scaled = convlstm_model.predict(
    [X_test_conv, S_test]
)

y_pred_convlstm = Y_scaler.inverse_transform(y_pred_scaled)
y_true_convlstm = Y_scaler.inverse_transform(y_test)


# =====================================================
# 6. IF YOU USED LOG-RAIN TRANSFORMATION
# =====================================================

rain_idx = targets.index("Rain")

# Get maximum predicted/true log-rain value
max_log_rain = max(
    np.nanmax(y_pred_convlstm[:, rain_idx]),
    np.nanmax(y_true_convlstm[:, rain_idx])
)

# Clip values before expm1 to avoid overflow
y_pred_convlstm[:, rain_idx] = np.clip(
    y_pred_convlstm[:, rain_idx],
    0,
    max_log_rain
)

y_true_convlstm[:, rain_idx] = np.clip(
    y_true_convlstm[:, rain_idx],
    0,
    max_log_rain
)

# Convert back from log scale
y_pred_convlstm[:, rain_idx] = np.expm1(
    y_pred_convlstm[:, rain_idx]
)

y_true_convlstm[:, rain_idx] = np.expm1(
    y_true_convlstm[:, rain_idx]
)
# =====================================================
# 7. EVALUATION
# =====================================================

convlstm_results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(
        y_true_convlstm[:, i],
        y_pred_convlstm[:, i]
    )

    mse = mean_squared_error(
        y_true_convlstm[:, i],
        y_pred_convlstm[:, i]
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_true_convlstm[:, i],
        y_pred_convlstm[:, i]
    )

    convlstm_results.append({
        "Model": "ConvLSTM",
        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

convlstm_results_df = pd.DataFrame(convlstm_results)

print(convlstm_results_df)

convlstm_results_df.to_excel(
    "ConvLSTM_Results.xlsx",
    index=False
)

print("Saved: ConvLSTM_Results.xlsx")

X_train_conv: (12389, 30, 1, 9, 1)
X_val_conv: (1664, 30, 1, 9, 1)
X_test_conv: (1668, 30, 1, 9, 1)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ convlstm_input      │ (None, 30, 1, 9,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_1       │ (None, 1, 9, 64)  │     50,176 │ convlstm_input[0… │
│ (ConvLSTM2D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_embedding   │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 576)       │          0 │ conv_lstm2d_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ station_embeddin… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 580)       │          0 │ flatten_1[0][0],  │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │     37,184 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 5)         │        325 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 87,705 (342.60 KB)

 Trainable params: 87,705 (342.60 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 28s 71ms/step - loss: 0.0230 - mae: 0.1072 - val_loss: 0.0123 - val_mae: 0.0709
Epoch 2/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 28s 73ms/step - loss: 0.0140 - mae: 0.0816 - val_loss: 0.0107 - val_mae: 0.0670
Epoch 3/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 29s 74ms/step - loss: 0.0123 - mae: 0.0742 - val_loss: 0.0104 - val_mae: 0.0624
Epoch 4/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step - loss: 0.0115 - mae: 0.0707 - val_loss: 0.0101 - val_mae: 0.0640
Epoch 5/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step - loss: 0.0110 - mae: 0.0682 - val_loss: 0.0101 - val_mae: 0.0638
Epoch 6/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step - loss: 0.0107 - mae: 0.0669 - val_loss: 0.0096 - val_mae: 0.0614
Epoch 7/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 30s 76ms/step - loss: 0.0105 - mae: 0.0661 - val_loss: 0.0096 - val_mae: 0.0602
Epoch 8/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 29s 76ms/step - loss: 0.0104 - mae: 0.0653 - val_loss: 0.0091 - val_mae: 0.0565
Epoch 9/100
388/388 ━━━━━━━━━━━━

In [15]:
from tensorflow.keras.layers import (
    Input, LSTM, Dense,
    Embedding, GlobalMaxPooling1D,
    Concatenate, Dropout
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# =====================================================
# LSTM ONLY MODEL
# =====================================================

seq_input = Input(
    shape=(30, num_features),
    name="sequence_input"
)

x = LSTM(
    64,
    return_sequences=False,
    name="lstm_layer"
)(seq_input)

station_input = Input(
    shape=(1,),
    name="station_input"
)

s = Embedding(
    input_dim=num_stations,
    output_dim=4,
    name="station_embedding"
)(station_input)

s = GlobalMaxPooling1D()(s)

combined = Concatenate()([x, s])

z = Dense(
    64,
    activation="relu"
)(combined)

z = Dropout(0.2)(z)

output = Dense(
    num_targets,
    activation="linear"
)(z)

lstm_model = Model(
    inputs=[seq_input, station_input],
    outputs=output
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = lstm_model.fit(
    [X_train, S_train],
    y_train,
    validation_data=([X_val, S_val], y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
y_pred_scaled = lstm_model.predict(
    [X_test, S_test]
)

y_pred = Y_scaler.inverse_transform(y_pred_scaled)
y_true = Y_scaler.inverse_transform(y_test)

y_pred = Y_scaler.inverse_transform(y_pred_scaled)
y_true = Y_scaler.inverse_transform(y_test)

results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
    mse = mean_squared_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true[:, i], y_pred[:, i])

    results.append({
        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

print(pd.DataFrame(results))

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequence_input      │ (None, 30, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_embedding   │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_layer (LSTM)   │ (None, 64)        │     18,944 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ station_embeddin… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 68)        │          0 │ lstm_layer[0][0], │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      4,416 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 5)         │        325 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,705 (92.60 KB)

 Trainable params: 23,705 (92.60 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0233 - mae: 0.1075 - val_loss: 0.0126 - val_mae: 0.0720
Epoch 2/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0136 - mae: 0.0802 - val_loss: 0.0113 - val_mae: 0.0673
Epoch 3/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0122 - mae: 0.0741 - val_loss: 0.0103 - val_mae: 0.0638
Epoch 4/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0113 - mae: 0.0704 - val_loss: 0.0099 - val_mae: 0.0606
Epoch 5/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0107 - mae: 0.0672 - val_loss: 0.0096 - val_mae: 0.0610
Epoch 6/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0104 - mae: 0.0654 - val_loss: 0.0103 - val_mae: 0.0645
Epoch 7/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0101 - mae: 0.0642 - val_loss: 0.0093 - val_mae: 0.0569
Epoch 8/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0099 - mae: 0.0629 - val_loss: 0.0093 - val_mae: 0.0564
Epoch 9/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/

In [19]:
from tensorflow.keras.layers import (
    Input, Conv1D, Dense,
    Embedding, GlobalMaxPooling1D,
    Concatenate, Dropout
)

from tensorflow.keras.models import Model
import tensorflow as tf

# =====================================================
# CNN ONLY MODEL
# =====================================================

seq_input = Input(
    shape=(30, num_features),
    name="sequence_input"
)

x = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu",
    name="conv1d_layer"
)(seq_input)

x = GlobalMaxPooling1D()(x)

station_input = Input(
    shape=(1,),
    name="station_input"
)

s = Embedding(
    input_dim=num_stations,
    output_dim=4,
    name="station_embedding"
)(station_input)

s = GlobalMaxPooling1D()(s)

combined = Concatenate()([x, s])

z = Dense(
    64,
    activation="relu"
)(combined)

z = Dropout(0.2)(z)

output = Dense(
    num_targets,
    activation="linear"
)(z)

cnn_model = Model(
    inputs=[seq_input, station_input],
    outputs=output
)

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

cnn_model.summary()
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = cnn_model.fit(
    [X_train, S_train],
    y_train,
    validation_data=([X_val, S_val], y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

y_pred_scaled_cnn = cnn_model.predict(
    [X_test, S_test]
)

y_pred = Y_scaler.inverse_transform(y_pred_scaled)
y_true = Y_scaler.inverse_transform(y_test)

results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
    mse = mean_squared_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true[:, i], y_pred[:, i])

    results.append({
        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

print(pd.DataFrame(results))

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_layer        │ (None, 28, 64)    │      1,792 │ sequence_input[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_embedding   │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_layer[0][… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ station_embeddin… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 68)        │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64)        │      4,416 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 5)         │        325 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,553 (25.60 KB)

 Trainable params: 6,553 (25.60 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0337 - mae: 0.1283 - val_loss: 0.0155 - val_mae: 0.0829
Epoch 2/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0186 - mae: 0.0955 - val_loss: 0.0150 - val_mae: 0.0787
Epoch 3/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0171 - mae: 0.0898 - val_loss: 0.0152 - val_mae: 0.0799
Epoch 4/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 998us/step - loss: 0.0159 - mae: 0.0853 - val_loss: 0.0156 - val_mae: 0.0792
Epoch 5/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0154 - mae: 0.0827 - val_loss: 0.0153 - val_mae: 0.0800
Epoch 6/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0150 - mae: 0.0811 - val_loss: 0.0148 - val_mae: 0.0765
Epoch 7/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0146 - mae: 0.0792 - val_loss: 0.0145 - val_mae: 0.0741
Epoch 8/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0143 - mae: 0.0782 - val_loss: 0.0140 - val_mae: 0.0734
Epoch 9/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 1m

In [17]:
print(lstm_model.summary())

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequence_input      │ (None, 30, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_embedding   │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_layer (LSTM)   │ (None, 64)        │     18,944 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ station_embeddin… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 68)        │          0 │ lstm_layer[0][0], │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      4,416 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 5)         │        325 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 71,117 (277.80 KB)

 Trainable params: 23,705 (92.60 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 47,412 (185.21 KB)

None


In [20]:
print(cnn_model.summary())

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_layer        │ (None, 28, 64)    │      1,792 │ sequence_input[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_embedding   │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_layer[0][… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ station_embeddin… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 68)        │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64)        │      4,416 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 5)         │        325 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,661 (76.80 KB)

 Trainable params: 6,553 (25.60 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 13,108 (51.21 KB)

None


In [22]:
y_pred_scaled_lstm = lstm_model.predict(
    [X_test, S_test]
)

print(y_pred_scaled_lstm.shape)

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
(1668, 5)


In [23]:
y_pred_scaled_cnn = cnn_model.predict(
    [X_test, S_test]
)

print(y_pred_scaled_cnn.shape)

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 920us/step
(1668, 5)


In [24]:
import numpy as np

print(
    np.allclose(
        y_pred_scaled_lstm,
        y_pred_scaled_cnn
    )
)

False


In [25]:
y_pred_lstm = Y_scaler.inverse_transform(y_pred_scaled_lstm)
y_true = Y_scaler.inverse_transform(y_test)

In [26]:
y_pred_cnn = Y_scaler.inverse_transform(y_pred_scaled_cnn)

In [27]:
def evaluate_model(y_true, y_pred, model_name):
    results = []

    for i, target in enumerate(targets):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        mse = mean_squared_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true[:, i], y_pred[:, i])

        results.append({
            "Model": model_name,
            "Target": target,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

    return pd.DataFrame(results)

In [28]:
lstm_results = evaluate_model(y_true, y_pred_lstm, "LSTM")
cnn_results = evaluate_model(y_true, y_pred_cnn, "CNN")

print(lstm_results)
print(cnn_results)

  Model    Target       MAE         MSE       RMSE        R2
0  LSTM      Temp  1.276410    2.999675   1.731957  0.938602
1  LSTM        RH  8.278570  127.469864  11.290255  0.658097
2  LSTM      Rain  1.617428   13.929130   3.732175  0.115066
3  LSTM     PRESS  2.907682   15.523994   3.940050  0.994620
4  LSTM  SUNSHINE  1.109880    3.094266   1.759053  0.831103
  Model    Target        MAE         MSE       RMSE        R2
0   CNN      Temp   1.988086    6.887499   2.624405  0.859025
1   CNN        RH  11.698477  239.660508  15.480972  0.357177
2   CNN      Rain   1.551243   14.535922   3.812600  0.076516
3   CNN     PRESS   4.935729   41.856185   6.469636  0.985493
4   CNN  SUNSHINE   1.300621    3.863523   1.965585  0.789114
